# Fine-tuning DistilBERT for Text Classification

This notebook outlines the steps to fine-tune a DistilBERT model for a text classification task, using data prepared by your teammate, and then uploading the fine-tuned adapters to Hugging Face.

## 1. Setup: Install and Import Libraries

First, we need to install the `transformers` library for working with pre-trained models and `datasets` for efficient data handling. We'll also include `accelerate` for optimized training and `evaluate` for metric computation.

In [ ]:
!pip install -qqq transformers datasets accelerate evaluate peft "torchao>=0.16.0"

In [ ]:
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset, DatasetDict
import evaluate
import numpy as np
from peft import get_peft_model, LoraConfig, TaskType

## 2. Data Loading and Preprocessing

This section is crucial. You'll need to load your data into a format that can be processed by the `datasets` library. Typically, this involves loading a CSV, JSON, or similar file that contains your text and corresponding labels.

**Action Required:** Please adapt the `load_your_data` function below to correctly load the data from your teammate's notebook. Look at `https://github.com/NateyLB/CMPE255Project/blob/main/notebooks/02_Data_Preprocessing.ipynb` to understand their data structure and output format. Ensure your data has 'text' and 'label' columns, or adjust the code accordingly.

For demonstration, I've included a placeholder for dummy data.

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
from datasets import Dataset, DatasetDict

def load_your_data():
    # Load the preprocessed data using pyarrow and convert to pandas
    try:
        df_train = pq.read_table('train.parquet').to_pandas()
        df_test = pq.read_table('test.parquet').to_pandas()
    except Exception as e:
        print(f"Error loading parquet files using pyarrow: {e}")
        raise # Re-raise the exception if loading fails

    # Rename 'cleaned_text' to 'text' to match the tokenizer's expectation
    df_train = df_train.rename(columns={'cleaned_text': 'text'})
    df_test = df_test.rename(columns={'cleaned_text': 'text'})

    # Convert pandas DataFrames to Hugging Face Datasets
    train_dataset = Dataset.from_pandas(df_train)
    test_dataset = Dataset.from_pandas(df_test)

    # Create a DatasetDict
    # For this example, we'll use the provided test set as both validation and final test
    dataset_dict = DatasetDict({
        'train': train_dataset,
        'validation': test_dataset.train_test_split(test_size=0.5, seed=42)['train'], # Use half of test for validation
        'test': test_dataset.train_test_split(test_size=0.5, seed=42)['test'] # Use other half for final test
    })

    return dataset_dict

# Load the dataset using the function above
dataset_dict = load_your_data()

print("Dataset loaded and split:")
print(dataset_dict)
print("Example from training set:")
print(dataset_dict['train'][0])

# Automatically map string labels to integer IDs
unique_labels = sorted(list(set(dataset_dict['train']['label'])))
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}

print(f"\nLabel mapping created: {label2id}")

# Apply the mapping to ensure labels are integers
dataset_dict = dataset_dict.map(lambda examples: {'label': [label2id[l] for l in examples['label']]}, batched=True)
print("Labels successfully converted to integers.")

## 3. Tokenization

To prepare the text data for DistilBERT, we need to convert it into a format that the model can understand. This involves tokenizing the text and mapping tokens to their corresponding IDs. We'll use the `DistilBertTokenizerFast` for this purpose.

**Important:** Ensure your text column is named 'text' and label column is named 'label' in your `dataset_dict`. If not, you might need to adjust the `tokenize_function` or rename your columns beforehand.

In [ ]:
print("\nFeatures in the training dataset:")
print(dataset_dict['train'].features)

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize_function(examples):
    # Changed 'text' to 'statement' to match the dataset's actual column name
    return tokenizer(examples['statement'], truncation=True)

tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)

print("Tokenization complete!")
print("Example tokenized entry from training set:")
print(tokenized_datasets['train'][0])

## 4. Model Loading and Metrics Definition

We'll load the pre-trained `DistilBertForSequenceClassification` model and define a function to compute evaluation metrics during training, specifically accuracy.

In [ ]:
num_labels = len(unique_labels) # Determine number of unique labels from our new dataset
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# Configure LoRA
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"] # Target attention blocks in DistilBERT
)

# Wrap the base model with the LoRA configuration
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return evaluate.load('accuracy').compute(predictions=predictions, references=labels)

print(f"\nModel loaded with {num_labels} labels and wrapped with LoRA adapters.")

## 5. Configure Training Arguments and Trainer

Here we define the `TrainingArguments`, which specify hyperparameters and settings for the training process. Then, we initialize the `Trainer` with our model, training arguments, datasets, tokenizer, and metrics function.

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch", # Changed from evaluation_strategy
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1, # Only save the last checkpoint
    # load_best_model_as_init_checkpoint=True, # Removed this argument
    report_to="none" # Disable experiment tracking for simplicity
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    # tokenizer=tokenizer, # Removed this argument
    compute_metrics=compute_metrics,
)

print("Trainer initialized with training arguments.")

## 6. Train the Model

Now, let's start the fine-tuning process. This might take some time depending on your dataset size and hardware.

In [ ]:
from transformers import DataCollatorWithPadding

print("Starting model training...")

# Add a data collator to pad sequences dynamically to the batch's max length
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
trainer.data_collator = data_collator

trainer.train()
print("Training complete!")

# Evaluate the model on the test set
results = trainer.evaluate(tokenized_datasets['test'])
print("\nEvaluation results on test set:")
print(results)

## 7. Upload Fine-tuned Adapters to Hugging Face

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
model.save_pretrained("distilBERT_mentalhealth_detection")  # Local saving
tokenizer.save_pretrained("distilBERT_mentalhealth_detection")
model.push_to_hub("NathanSJSU01/distilBERT_mentalhealth_detection", token = HF_TOKEN) # Online saving
tokenizer.push_to_hub("NathanSJSU01/distilBERT_mentalhealth_detection", token = HF_TOKEN) # Online saving